In [23]:
# download tiny shakespeare
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "shakespeare.txt"
)

text = open('shakespeare.txt').read()
print(f"Total characters: {len(text)}")
print(f"Sample: {text[:100]}")


Total characters: 1115394
Sample: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [31]:
name = "shankar"

set(name)

{'a', 'h', 'k', 'n', 'r', 's'}

In [24]:
import numpy as np

In [ ]:
# step 1 — vocabulary
chars      = sorted(set(text))
vocab_size = len(chars)
stoi       = {char: index for index, char in enumerate(chars)}
itos       = {index: char for index, char in enumerate(chars)}

# step 2 — encode and decode
def encode(s):
    "takes a string and returns a list of integers"
    ids = [stoi[char] for char in s]
    return ids    

def decode(ids):
    "takes a list of ids and return string"
    char = [itos[id] for id in ids]
    s = "".join(char)
    return s

# step 3 — encode full dataset
data = np.array(encode(text))

print(f"Data shape: {data.shape}")
print(f"First 20 tokens: {data[:20]}")
print(f"Decoded back: {decode(data[:20])}")

# step 4 — create training pairs
seq_len = 64

rng = np.random.default_rng(42)

def get_batch(data, seq_len, batch_size):
    # randomly sample batch_size starting positions
    # X: (batch_size, seq_len)
    # y: (batch_size, seq_len) — shifted by one
    data_size = len(data)
    max_starting_position = data_size - seq_len - 1 
    batch_seq = rng.integers(0, max_starting_position, size=batch_size)
    ix = batch_seq[:, None] + np.arange(seq_len) # fancy indexing: Indexing with an array 
    """
    data[ix] means — "for every number in ix, go to that position in data and grab the value
    there." The result has the same shape as ix.
    """
    X = data[ix]
    y = data[ix + 1] # here data[ix] + 1 will give value absolute to indexing to the position
    # but [ix + 1] will give value relative in the current position. 
    return X, y

X, y = get_batch(data, seq_len=8, batch_size=4)
print(decode(X[0].tolist()))
print(decode(y[0].tolist()))


1115394
Data shape: (1115394,)
First 20 tokens: [18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56]
Decoded back: First Citizen:
Befor
1115385
e of us:
 of us: 


In [34]:
d_model  = 128
n_heads  = 4
n_layers = 4
d_ff     = 512
seq_len  = 64

# token embedding
W_embed = rng.standard_normal((vocab_size, d_model)) * 0.02

# Positional Embedding
W_pos = rng.standard_normal((seq_len, d_model)) * 0.02

def embed(X):
    token_embed = W_embed[X]
    pos_embed = W_pos

    # combine and return 
    return token_embed + pos_embed  # (batch_size, seq_len, d_model)

X, y = get_batch(data, seq_len, batch_size=4)
embedded = embed(X)
print(embedded.shape)


1115329
(4, 64, 128)
